# PesaPap Growth Analytics

This notebook builds a local SQLite database from the three seed-77 CSV files, executes the validation query and SQL questions a–h, then presents KPI, RFM, cohort-retention and onboarding experiment analysis. Currency values are Kenyan shillings (KES). The data are synthetic.

## Data model and integrity

- `users`: one row per registered user; primary key `user_id`.
- `transactions`: one row per payment; primary key `txn_id`, foreign key `user_id` → `users.user_id`.
- `experiment`: one row per assigned user; primary key and foreign key `user_id` → `users.user_id`.

The validation query identifies transaction dates earlier than signup. Such records are retained unchanged in source tables for auditability but excluded from behavioral KPIs, RFM, cohorts and experiment-linked customer-value summaries: a user cannot transact in the app before registering. The SQL assignment outputs below document the source-table query results; the pandas analysis explicitly applies this eligibility rule.

In [ ]:
from pathlib import Path
import sqlite3
import re
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare
from IPython.display import display

BASE = Path.cwd()
DB_PATH = BASE / "pesapap.db"
SQL_PATH = BASE / "sql" / "queries.sql"

users = pd.read_csv(BASE / "users.csv", parse_dates=["signup_date"])
transactions = pd.read_csv(BASE / "transactions.csv", parse_dates=["txn_date"])
experiment = pd.read_csv(BASE / "experiment.csv")

if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
CREATE TABLE users (
  user_id INTEGER PRIMARY KEY,
  signup_date TEXT NOT NULL,
  county TEXT NOT NULL,
  acquisition_channel TEXT NOT NULL
);
CREATE TABLE transactions (
  txn_id INTEGER PRIMARY KEY,
  user_id INTEGER NOT NULL REFERENCES users(user_id),
  txn_date TEXT NOT NULL,
  product TEXT NOT NULL,
  amount REAL NOT NULL
);
CREATE TABLE experiment (
  user_id INTEGER PRIMARY KEY REFERENCES users(user_id),
  "group" TEXT NOT NULL CHECK ("group" IN ('A', 'B')),
  converted INTEGER NOT NULL CHECK (converted IN (0, 1))
);
""")
users_for_db = users.copy()
transactions_for_db = transactions.copy()
users_for_db["signup_date"] = users_for_db["signup_date"].dt.strftime("%Y-%m-%d")
transactions_for_db["txn_date"] = transactions_for_db["txn_date"].dt.strftime("%Y-%m-%d")
users_for_db.to_sql("users", conn, if_exists="append", index=False)
transactions_for_db.to_sql("transactions", conn, if_exists="append", index=False)
experiment.to_sql("experiment", conn, if_exists="append", index=False)
conn.commit()

table_check = pd.read_sql("SELECT 'users' AS table_name, COUNT(*) AS rows FROM users UNION ALL SELECT 'transactions', COUNT(*) FROM transactions UNION ALL SELECT 'experiment', COUNT(*) FROM experiment", conn)
foreign_key_check = pd.read_sql("PRAGMA foreign_key_check", conn)
print(f"Created {DB_PATH.name}; foreign-key violations: {len(foreign_key_check)}")
display(table_check)

## SQL validation and business questions

Each statement is loaded from `sql/queries.sql`; the leading SQL comment is its business question. Every query is executed through `pd.read_sql` so the result tables are visible in the notebook. The validation result is interpreted separately because it is a data-quality check, not a valid-user activity KPI.

In [ ]:
sql_text = SQL_PATH.read_text(encoding="utf-8")
sql_blocks = [block.strip() for block in re.split(r"(?m)(?=^-- )", sql_text) if block.strip()]
query_results = {}
for block in sql_blocks:
    lines = block.splitlines()
    question = lines[0].removeprefix("-- ").strip()
    query = "\n".join(line for line in lines[1:] if not line.lstrip().startswith("--")).strip()
    if query:
        result = pd.read_sql(query, conn)
        query_results[question] = result
        print(question)
        display(result)

validation_question = next(q for q in query_results if q.startswith("Validation:"))
pre_signup_count = int(query_results[validation_question].iloc[0, 0])
print(f"Data-quality finding: {pre_signup_count:,} transactions predate signup. They remain auditable in SQLite, but are excluded from user-journey analytics.")

## KPI scorecard

All KPIs below use eligible transactions only (`txn_date >= signup_date`). Month-over-month growth compares the latest observed calendar month with the immediately preceding month. The generator provides activity only through part of May/June depending on transaction generation; the latest month may therefore be partial and its growth is provisional.

In [ ]:
joined = transactions.merge(users[["user_id", "signup_date"]], on="user_id", validate="many_to_one")
eligible = joined.loc[joined["txn_date"] >= joined["signup_date"]].copy()
eligible["month"] = eligible["txn_date"].dt.to_period("M").astype(str)
monthly_kpis = eligible.groupby("month").agg(transaction_value_kes=("amount", "sum"), transactions=("txn_id", "count"), active_users=("user_id", "nunique"))
monthly_kpis["mom_value_growth_pct"] = monthly_kpis["transaction_value_kes"].pct_change() * 100
total_value = float(eligible["amount"].sum())
transaction_count = int(eligible["txn_id"].nunique())
active_users = int(eligible["user_id"].nunique())
average_transaction_value = total_value / transaction_count
average_transactions_per_active_user = transaction_count / active_users
latest_mom_growth = float(monthly_kpis["mom_value_growth_pct"].iloc[-1])

kpi_values = [
    ("Total transaction value", "Sum of KES transaction amounts occurring on or after the user’s signup date.", "SUM(amount) over eligible transactions", total_value),
    ("Transactions", "Count of unique eligible transaction IDs.", "COUNT(DISTINCT txn_id)", transaction_count),
    ("Active users", "Distinct users with at least one eligible transaction in the full observation window.", "COUNT(DISTINCT user_id)", active_users),
    ("Average transaction value", "Mean KES value per eligible transaction.", "SUM(amount) / COUNT(DISTINCT txn_id)", average_transaction_value),
    ("Transactions per active user", "Average eligible transaction frequency among observed active users.", "eligible transaction count / active users", average_transactions_per_active_user),
    ("Latest month-over-month value growth", "Percentage change in eligible transaction value versus the preceding calendar month. Latest month can be partial.", "(current month value / previous month value − 1) × 100", latest_mom_growth),
]
kpi_scorecard = pd.DataFrame(kpi_values, columns=["name", "definition", "calculation", "value"])
display(kpi_scorecard)
print("Monthly KPI trend (latest period may be partial):")
display(monthly_kpis.round(2))

## RFM segments and monthly cohort retention

RFM recency is measured in days before the last eligible transaction date plus one day; frequency is eligible transaction count and monetary is eligible value. Quartile scores are assigned with rank-based quantiles (R is reversed so a recent customer scores higher). Segments prioritize high recency/frequency as Champions/Loyal and high frequency but stale recency as At Risk High Frequency. Cohort retention uses signup month, the first observed eligible transaction month as cohort month 0, and distinct active users divided by cohort-month-0 active users. Cells after the latest fully observed month are labeled `INCOMPLETE` rather than interpreted as churn.

In [ ]:
snapshot_date = eligible["txn_date"].max() + pd.Timedelta(days=1)
rfm = eligible.groupby("user_id").agg(last_txn=("txn_date", "max"), frequency=("txn_id", "nunique"), monetary_kes=("amount", "sum"))
rfm["recency_days"] = (snapshot_date - rfm["last_txn"]).dt.days
rfm["R"] = pd.qcut(rfm["recency_days"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary_kes"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [(rfm.R >= 3) & (rfm.F >= 3), (rfm.R <= 2) & (rfm.F >= 3), (rfm.R >= 3) & (rfm.F <= 2)],
    ["Champions/Loyal", "At Risk High Frequency", "New/Potential"],
    default="Hibernating/Other",
)
rfm_summary = rfm.groupby("segment", observed=False).agg(users=("segment", "size"), mean_value_kes=("monetary_kes", "mean"), total_value_kes=("monetary_kes", "sum")).sort_values("total_value_kes", ascending=False)
print("RFM segment definitions: R=recency quartile (4 most recent); F=frequency quartile; M=monetary quartile.")
display(rfm_summary.round(2))

user_cohorts = users[["user_id", "signup_date"]].copy()
user_cohorts["cohort_month"] = user_cohorts["signup_date"].dt.to_period("M")
cohort_activity = eligible[["user_id", "txn_date"]].merge(user_cohorts[["user_id", "cohort_month"]], on="user_id", validate="many_to_one")
cohort_activity["activity_month"] = cohort_activity["txn_date"].dt.to_period("M")
cohort_activity["cohort_index"] = (cohort_activity.activity_month.dt.year - cohort_activity.cohort_month.dt.year) * 12 + cohort_activity.activity_month.dt.month - cohort_activity.cohort_month.dt.month
cohort_activity = cohort_activity[cohort_activity["cohort_index"] >= 0]
cohort_size = user_cohorts.groupby("cohort_month")["user_id"].nunique()
active_counts = cohort_activity.groupby(["cohort_month", "cohort_index"])["user_id"].nunique().unstack("cohort_index")
active_counts = active_counts.reindex(index=cohort_size.index, columns=range(6), fill_value=0)
retention = active_counts.div(cohort_size, axis=0)
latest_data_month = eligible["txn_date"].max().to_period("M")
latest_complete_month = latest_data_month - 1
retention_display = retention.copy().astype(object)
for cohort_month in retention.index:
    for month_index in retention.columns:
        observed_month = cohort_month + int(month_index)
        if observed_month > latest_complete_month:
            retention_display.loc[cohort_month, month_index] = "INCOMPLETE"
        else:
            retention_display.loc[cohort_month, month_index] = f"{retention.loc[cohort_month, month_index]:.1%}"
retention_display.index = retention_display.index.astype(str)
retention_display.columns = [f"Month {int(column)}" for column in retention_display.columns]
print(f"Cohort table: denominator is all sign-ups in each signup-month cohort; latest eligible data month: {latest_data_month}; latest fully observed month: {latest_complete_month}.")
display(retention_display)

target_segment = "At Risk High Frequency" if "At Risk High Frequency" in rfm_summary.index else rfm_summary.index[0]
print(f"First retention experiment target: {target_segment}. These users have repeat transaction history but have gone relatively quiet; a measured reactivation offer can test recoverable value before spending on broad acquisition.")

## RFM segments and monthly cohort retention

RFM recency is measured in days before the last eligible transaction date plus one day; frequency is eligible transaction count and monetary is eligible value. Quartile scores are assigned with rank-based quantiles (R is reversed so a recent customer scores higher). Segments prioritize high recency/frequency as Champions/Loyal and high frequency but stale recency as At Risk High Frequency. Cohort retention uses signup month and all registered users in that month as the denominator; month 0 therefore reflects the share transacting in their signup month. Cells after the latest fully observed month are labeled `INCOMPLETE` rather than interpreted as churn.

In [ ]:
group_sizes = experiment["group"].value_counts().reindex(["A", "B"])
balance_test = chisquare(group_sizes.to_numpy(), f_exp=np.repeat(group_sizes.sum() / 2, 2))
experiment_summary = experiment.groupby("group")["converted"].agg(conversions="sum", users="count", conversion_rate="mean").reindex(["A", "B"])
p_a = float(experiment_summary.loc["A", "conversion_rate"])
p_b = float(experiment_summary.loc["B", "conversion_rate"])
n_a = int(experiment_summary.loc["A", "users"])
n_b = int(experiment_summary.loc["B", "users"])
absolute_lift = p_b - p_a
relative_lift = absolute_lift / p_a
standard_error = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low, ci_high = absolute_lift - 1.96 * standard_error, absolute_lift + 1.96 * standard_error
conversion_table = pd.crosstab(experiment["group"], experiment["converted"]).reindex(index=["A", "B"], columns=[0, 1], fill_value=0)
chi2, p_value, dof, expected = chi2_contingency(conversion_table, correction=False)

print(f"Allocation balance: A={n_a:,}, B={n_b:,}; 50/50 chi-square p={balance_test.pvalue:.4f}")
display(experiment_summary.assign(conversion_rate_pct=experiment_summary.conversion_rate * 100))
print(f"Absolute lift (B - A): {absolute_lift:.3%} ({absolute_lift * 100:.2f} percentage points)")
print(f"Relative lift: {relative_lift:.1%}")
print(f"95% CI for absolute lift: [{ci_low:.3%}, {ci_high:.3%}]")
print(f"Pearson chi-square: {chi2:.3f}; df={dof}; p={p_value:.4f}")
if p_value < 0.05 and ci_low > 0:
    experiment_interpretation = "Evidence of higher conversion in B at the 5% level; consider a monitored phased rollout."
elif p_value < 0.05 and ci_high < 0:
    experiment_interpretation = "Evidence that B reduces conversion at the 5% level; do not roll it out."
else:
    experiment_interpretation = "The data do not establish a conversion difference at the 5% level; retain A or continue testing."
print(experiment_interpretation)

ASSUMED_MONTHLY_SIGNUPS = 3000
estimated_incremental_conversions_per_month = ASSUMED_MONTHLY_SIGNUPS * absolute_lift
impact_ci = (ASSUMED_MONTHLY_SIGNUPS * ci_low, ASSUMED_MONTHLY_SIGNUPS * ci_high)
print(f"Illustrative impact at {ASSUMED_MONTHLY_SIGNUPS:,} monthly signups: {estimated_incremental_conversions_per_month:.1f} incremental conversions/month (95% interval {impact_ci[0]:.1f} to {impact_ci[1]:.1f}), using the observed absolute lift. This is a scenario, not a forecast.")

## Decision guardrails

The experiment measures signup conversion only. Before a broad rollout, verify balance, review statistical uncertainty, and monitor downstream eligible transaction activity, value and retention. A conversion lift does not prove incremental revenue or long-term customer value. See `reports/growth_memo.md` for the one-page leadership memo.

## Key results from the seed-77 run

The notebook code above was executed from top to bottom in Python 3.12. The following deterministic results are surfaced here for quick review; rerunning the notebook recomputes them from the CSVs.

### Validation and SQL answers

- **Pre-signup transactions:** 3,737 of 12,000. These records remain in the database for audit, but are excluded from customer behavior analyses because activity predating registration is not a valid user journey.
- **Value by product (raw SQL source totals):** Send Money KES 5,639,308 (4,161 transactions); Airtime KES 4,708,104 (3,662); Bill Pay KES 3,224,392 (2,388); Merchant Pay KES 2,407,505 (1,789).
- **Top counties by raw transaction value:** Mombasa KES 3,464,271; Nairobi KES 3,403,364; Nakuru KES 3,138,079; Kisumu KES 3,059,055; Eldoret KES 2,914,540.
- **Raw monthly value:** Jan KES 1,733,121; Feb KES 2,993,105; Mar KES 3,199,125; Apr KES 3,262,447; May KES 3,365,002; Jun KES 1,426,509. June is partial (through June 13).
- **Acquisition channel raw value:** Search KES 4,279,089; Agent KES 3,954,250; Social KES 3,926,296; Referral KES 3,819,674.
- The SQL cells also display the top ten users by `RANK()`, products above the KES 1,300 average threshold, High/Medium/Low lifetime-value counts, and monthly active-user changes using `LAG()`.

### KPI scorecard (eligible post-signup transactions)

| KPI | Result |
| --- | ---: |
| Total transaction value | KES 10,956,778 |
| Transactions | 8,263 |
| Active users | 2,716 |
| Average transaction value | KES 1,326.00 |
| Transactions per active user | 3.04 |
| June value growth vs May | -57.61% (partial June; not a like-for-like monthly comparison) |

### RFM segments and retention

| RFM segment | Users | Mean eligible value | Total eligible value |
| --- | ---: | ---: | ---: |
| Champions/Loyal | 814 | KES 5,967.11 | KES 4,857,231 |
| At Risk High Frequency | 544 | KES 5,510.19 | KES 2,997,545 |
| Hibernating/Other | 814 | KES 2,151.01 | KES 1,750,919 |
| New/Potential | 544 | KES 2,483.61 | KES 1,351,083 |

Cohort retention uses all sign-ups as each cohort denominator; future months and partial June observations are marked `INCOMPLETE` in the computed table rather than treated as zero retention.

### Experiment decision

| Group | Converted | Assigned | Conversion |
| --- | ---: | ---: | ---: |
| A | 151 | 1,443 | 10.46% |
| B | 215 | 1,557 | 13.81% |

Observed lift is **+3.34 percentage points** (**+32.0% relative**); the 95% approximate confidence interval is **+1.01 to +5.67 percentage points**, with Pearson chi-square **p = 0.0052**. A 50/50 allocation balance check yields **p = 0.0374** (A 48.1%, B 51.9%), a modest realized imbalance to note. At 3,000 assumed monthly sign-ups, the sample lift implies about **100 additional conversions/month** (approximate interval 30–170). Recommendation: monitored phased rollout and a randomized holdout, followed by validation on production data.